In [1]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# PARAMETERS
# ============================================================

num_episodes = 25_000

gamma = 0.99

# This is the "overall" learning-rate parameter.
# Since num_tilings features are active at once,
# we divide it by num_tilings during each update.
alpha = 0.5

epsilon_start = 1.0
epsilon_min = 0.01
epsilon_decay = 0.00025

num_tilings = 8
tiles_per_dim = 8

seed = 42

rng = np.random.default_rng(seed)


# ============================================================
# ENVIRONMENT
# ============================================================

env = gym.make("MountainCar-v0")

state_low = env.observation_space.low
state_high = env.observation_space.high

num_actions = env.action_space.n

print("State low:", state_low)
print("State high:", state_high)
print("Number of actions:", num_actions)

State low: [-1.2  -0.07]
State high: [0.6  0.07]
Number of actions: 3


In [ ]:
# ============================================================
# TILE CODING
# ============================================================

num_features_per_tiling = tiles_per_dim * tiles_per_dim

num_features = num_tilings * num_features_per_tiling


def get_active_features(state):
    """
    Return the indices of the active tile features for a
    continuous MountainCar state.

    Exactly one feature is active in each tiling.
    """

    position, velocity = state

    active_features = []

    # Width of one tile in physical state coordinates
    tile_width_position = (
        state_high[0] - state_low[0]
    ) / tiles_per_dim

    tile_width_velocity = (
        state_high[1] - state_low[1]
    ) / tiles_per_dim

    for tiling in range(num_tilings):

        # Different offsets for the two dimensions
        offset_fraction_position = tiling / num_tilings

        offset_fraction_velocity = (
            (2 * tiling + 1) % num_tilings
        ) / num_tilings

        offset_position = (
            offset_fraction_position
            * tile_width_position
        )

        offset_velocity = (
            offset_fraction_velocity
            * tile_width_velocity
        )

        # Determine tile coordinates in this tiling
        position_index = int(
            np.floor(
                (
                    position
                    - state_low[0]
                    + offset_position
                )
                / tile_width_position
            )
        )

        velocity_index = int(
            np.floor(
                (
                    velocity
                    - state_low[1]
                    + offset_velocity
                )
                / tile_width_velocity
            )
        )

        # Keep indices inside the grid
        position_index = np.clip(
            position_index,
            0,
            tiles_per_dim - 1
        )

        velocity_index = np.clip(
            velocity_index,
            0,
            tiles_per_dim - 1
        )

        # Convert the 2D tile coordinate into one local index
        local_index = (
            position_index * tiles_per_dim
            + velocity_index
        )

        # Each tiling owns a different part of the feature vector
        global_index = (
            tiling * num_features_per_tiling
            + local_index
        )

        active_features.append(global_index)

    return np.array(active_features, dtype=int)

In [ ]:
# ============================================================
# LINEAR FUNCTION APPROXIMATOR
# ============================================================

weights = np.zeros(
    (num_actions, num_features)
)

In [9]:
def q_values(state, action):
    active = get_active_features(state)

    return np.sum(
        weights[action, active]
    )

In [10]:
def choose_action(state, epsilon):

    if rng.random() < epsilon:
        return rng.integers(num_actions)

    values = q_values(state, action)

    max_actions = np.flatnonzero(
        values == np.max(values)
    )

    return rng.choice(max_actions)

In [11]:
def update_weights(
    state,
    action,
    reward,
    next_state,
    terminated
):

    active = get_active_features(state)

    current_q = np.sum(
        weights[action, active]
    )

    if terminated:

        target = reward

    else:

        next_q = q_values(next_state, action)

        target = (
            reward
            + gamma * np.max(next_q)
        )

    td_error = target - current_q

    # Because 8 features are simultaneously active,
    # scale the step size accordingly.
    effective_alpha = alpha / num_tilings

    weights[action, active] += (
        effective_alpha * td_error
    )

    return td_error

In [ ]:
# ============================================================
# TRAINING
# ============================================================

episode_lengths = []
episode_successes = []
td_errors = []


for episode in range(num_episodes):

    state, info = env.reset()

    epsilon = (
        epsilon_min
        +
        (epsilon_start - epsilon_min)
        * np.exp(-epsilon_decay * episode)
    )

    terminated = False
    truncated = False

    steps = 0

    while not (terminated or truncated):

        action = choose_action(
            state,
            epsilon
        )

        (
            next_state,
            reward,
            terminated,
            truncated,
            info
        ) = env.step(action)

        td_error = update_weights(
            state,
            action,
            reward,
            next_state,
            terminated
        )

        td_errors.append(abs(td_error))

        state = next_state

        steps += 1

    episode_lengths.append(steps)

    success = int(terminated)

    episode_successes.append(success)

    if (episode + 1) % 1000 == 0:

        recent_success_rate = np.mean(
            episode_successes[-1000:]
        )

        recent_length = np.mean(
            episode_lengths[-1000:]
        )

        print(
            f"Episode {episode + 1:5d} | "
            f"epsilon = {epsilon:.3f} | "
            f"success = {recent_success_rate:.3f} | "
            f"length = {recent_length:.1f}"
        )

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [ ]:
rolling_window = 500

success_array = np.array(
    episode_successes,
    dtype=float
)

kernel = np.ones(rolling_window) / rolling_window

rolling_success = np.convolve(
    success_array,
    kernel,
    mode="valid"
)

plt.figure(figsize=(10, 5))

plt.plot(
    np.arange(
        rolling_window - 1,
        num_episodes
    ),
    rolling_success
)

plt.xlabel("Episode")
plt.ylabel("Rolling success rate")
plt.title(
    f"MountainCar Function Approximation "
    f"({num_tilings} tilings)"
)

plt.grid()

plt.show()

ValueError: v cannot be empty

In [ ]:
rolling_window = 500

success_array = np.array(
    episode_successes,
    dtype=float
)

kernel = np.ones(rolling_window) / rolling_window

rolling_success = np.convolve(
    success_array,
    kernel,
    mode="valid"
)

plt.figure(figsize=(10, 5))

plt.plot(
    np.arange(
        rolling_window - 1,
        num_episodes
    ),
    rolling_success
)

plt.xlabel("Episode")
plt.ylabel("Rolling success rate")
plt.title(
    f"MountainCar Function Approximation "
    f"({num_tilings} tilings)"
)

plt.grid()

plt.show()

ValueError: v cannot be empty

In [ ]:
# ============================================================
# TESTING
# ============================================================

num_test_episodes = 1000

test_successes = 0
test_episode_lengths = []


for episode in range(num_test_episodes):

    state, info = env.reset()

    terminated = False
    truncated = False

    steps = 0

    while not (terminated or truncated):

        values = q_values(state)

        max_actions = np.flatnonzero(
            values == np.max(values)
        )

        action = rng.choice(max_actions)

        (
            next_state,
            reward,
            terminated,
            truncated,
            info
        ) = env.step(action)

        state = next_state
        steps += 1

    test_successes += int(terminated)

    test_episode_lengths.append(steps)


print(
    "Test success rate:",
    test_successes / num_test_episodes
)

print(
    "Average test episode length:",
    np.mean(test_episode_lengths)
)